# YT Collector — Whisper Transcriber

Polls Supabase for `whisper_processing` queue items, transcribes audio with Whisper, and POSTs results back to Vercel.

**Before running:** add these to Colab Secrets (key icon in left sidebar):
- `CALLBACK_SECRET`
- `CALLBACK_URL` — e.g. `https://your-app.vercel.app/api/whisper/callback`
- `SUPABASE_URL`
- `SUPABASE_SERVICE_ROLE_KEY`

Then run all cells. The last cell loops forever — interrupt the kernel to stop.

In [ ]:
!pip install -q openai-whisper requests

In [ ]:
import whisper
import requests
import tempfile
import os
import time
from google.colab import userdata

CALLBACK_SECRET = userdata.get('CALLBACK_SECRET')
CALLBACK_URL    = userdata.get('CALLBACK_URL')
SUPABASE_URL    = userdata.get('SUPABASE_URL')
SUPABASE_KEY    = userdata.get('SUPABASE_SERVICE_ROLE_KEY')

POLL_INTERVAL_SEC = 30

model = whisper.load_model('small')
print('Model loaded. Starting polling loop...')

In [ ]:
def format_transcript(segments: list) -> str:
    lines = []
    for seg in segments:
        start = int(seg['start'])
        minutes, seconds = divmod(start, 60)
        lines.append(f'[{minutes:02d}:{seconds:02d}] {seg["text"].strip()}')
    return '\n'.join(lines)


def fetch_next_job():
    resp = requests.get(
        f'{SUPABASE_URL}/rest/v1/queue',
        params={'status': 'eq.whisper_processing', 'order': 'created_at.asc', 'limit': 1},
        headers={'apikey': SUPABASE_KEY, 'Authorization': f'Bearer {SUPABASE_KEY}'}
    )
    items = resp.json()
    return items[0] if items else None


def fetch_audio_url(youtube_id: str) -> str | None:
    resp = requests.get(
        f'{SUPABASE_URL}/rest/v1/videos',
        params={'youtube_id': f'eq.{youtube_id}', 'select': 'audio_r2_url'},
        headers={'apikey': SUPABASE_KEY, 'Authorization': f'Bearer {SUPABASE_KEY}'}
    )
    items = resp.json()
    return items[0]['audio_r2_url'] if items else None


def mark_error(queue_id: str, error: str, retries: int):
    status = 'error_whisper' if retries >= 3 else 'whisper_processing'
    requests.patch(
        f'{SUPABASE_URL}/rest/v1/queue?id=eq.{queue_id}',
        json={'status': status, 'whisper_retries': retries, 'last_error': error},
        headers={'apikey': SUPABASE_KEY, 'Authorization': f'Bearer {SUPABASE_KEY}',
                 'Content-Type': 'application/json', 'Prefer': 'return=minimal'}
    )


def transcribe(job: dict):
    queue_id   = job['id']
    youtube_id = job['youtube_id']
    retries    = job.get('whisper_retries', 0) + 1

    audio_url = fetch_audio_url(youtube_id)
    if not audio_url:
        print(f'[{youtube_id}] No audio URL, skipping.')
        return

    print(f'[{youtube_id}] Downloading audio...')
    try:
        with tempfile.NamedTemporaryFile(suffix='.mp3', delete=False) as tmp:
            r = requests.get(audio_url, timeout=120)
            r.raise_for_status()
            tmp.write(r.content)
            tmp_path = tmp.name

        print(f'[{youtube_id}] Transcribing...')
        result = model.transcribe(tmp_path, fp16=True)
        os.unlink(tmp_path)

        transcript = format_transcript(result['segments'])
        print(f'[{youtube_id}] Done — {len(result["segments"])} segments. Sending callback...')

        resp = requests.post(
            CALLBACK_URL,
            json={'queue_id': queue_id, 'transcript': transcript},
            headers={'x-callback-secret': CALLBACK_SECRET},
            timeout=30
        )
        print(f'[{youtube_id}] Callback: {resp.status_code}')

    except Exception as e:
        print(f'[{youtube_id}] Error: {e}')
        mark_error(queue_id, str(e), retries)

In [ ]:
# Polling loop — runs until kernel is interrupted
while True:
    job = fetch_next_job()
    if job:
        transcribe(job)
    else:
        print('No jobs. Waiting...')
    time.sleep(POLL_INTERVAL_SEC)